# **YOLO - Detección y seguimiento de muñeca mediante distancias**
Pasos a seguir:
1. Detectar una muñeca con YOLO.
2. Realizar el seguimiento de la muñeca seleccionada mediante distancias.

Para llevar a cabo esta tarea, seleccionamos la primera muñeca izquierda (`ID=9`) detectada con YOLO. 

Utilizamos la variable `punto_base` para almacenar el punto de partida de cada frame y la variable `history_dist` para almacenar las distancias de las muñecas detectadas durante la reproducción del vídeo. Luego, si no se ha detectado ninguna muñeca aún, se almacena en `punto_base` la primera muñeca detectada. Seguidamente, se guardan las distancias entre los puntos detectados con YOLO y el punto de referencia. 

Una vez recorridos todos los puntos detectados, se ordenan las distancias de dichos puntos y se escoge el mejor resultado: el punto con la menor distancia. Luego, si aún no hay resultados o la máxima distancia es nula, se añade el resultado obtenido. En caso contrario, se comprueba primero que la distancia al punto base del nuevo resultado sea, como mucho, 1.8 veces la máxima distancia encontrada.

In [15]:
import torch
import cv2
import numpy as np
import math
from ultralytics import YOLO

# Model
model = YOLO('yolo11x-pose.pt')
#model=YOLO('yolov8n-pose.pt') más rápido
# Images
video = cv2.VideoCapture("videos/people2.mp4")

success, frame = video.read()
size=(frame.shape[1], frame.shape[0])
fourcc = cv2.VideoWriter_fourcc(*'DIVX')
video_out = cv2.VideoWriter('output/yolo_pose.mp4', fourcc, 30.0, size)

punto_base = None
history_dist = []

# Bucle a través de los fotogramas del video
while (success):
    puntos =[]
    dist_ptos = []
   
    # Ejecutar seguimiento YOLOv8 en el fotograma, persistiendo los rastreos entre fotogramas
    results = model.track(frame)[0]

    for r in results:
        kpts = r.keypoints
        nk = kpts.shape[1]
        for i in range(nk):
            if i==9: # muñeca izquierda
                keypoint=kpts.xy[0,i]    
                x, y = (int(keypoint[0])),(int(keypoint[1]))
                if punto_base is None:
                    punto_base = keypoint
                    cv2.circle(frame, (x,y),5,(0,0,255),-1)
                dist_temp = math.dist(punto_base, keypoint)
                if punto_base is not None:
                    dist_ptos.append((keypoint, dist_temp))
    print(dist_ptos)
    mejor = sorted(dist_ptos, key= lambda x: x[1])[0]
    print(history_dist)
    if len(history_dist) != 0 and sum(history_dist) != 0:
        print(f"Mejor: {mejor[1]}")
        print(f"max: {max(history_dist)}")
        if mejor[1] < max(history_dist) * 1.8:
            punto_base = mejor[0]
            history_dist.append(mejor[1])
            print(mejor)
            cv2.circle(frame, ((int(mejor[0][0])),(int(mejor[0][1]))),5,(0,0,255),-1)
    else:
        punto_base = mejor[0]
        history_dist.append(mejor[1])
        print(mejor)
        cv2.circle(frame, ((int(mejor[0][0])),(int(mejor[0][1]))),5,(0,0,255),-1)
                
    # Mostrar el fotograma anotado
    cv2.imshow('salida',frame)
    video_out.write(frame)
    success, frame = video.read()
    # Romper el bucle si se presiona 'q'
    if cv2.waitKey(1) & 0xFF == ord("q"):
         success = False
         break
    

# Liberar el objeto de captura de video y cerrar la ventana de visualización
video.release()
video_out.release()
cv2.destroyAllWindows()


0: 384x640 7 persons, 279.6ms
Speed: 0.8ms preprocess, 279.6ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)
[(tensor([449.9027, 494.2927]), 0.0), (tensor([973.1995, 505.4446]), 523.4155972519832), (tensor([ 71.5825, 562.6755]), 384.4507043742105), (tensor([775.7858, 665.6659]), 368.1964004587411), (tensor([508.8169, 342.8883]), 162.4628659589321), (tensor([212.4267, 346.5956]), 279.6592592500096), (tensor([1278.4751,  499.3896]), 828.5880944314175)]
[]
(tensor([449.9027, 494.2927]), 0.0)

0: 384x640 7 persons, 328.6ms
Speed: 1.0ms preprocess, 328.6ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
[(tensor([967.6624, 502.2855]), 517.8214249294863), (tensor([452.8474, 488.6693]), 6.3477874052729835), (tensor([771.2878, 654.4947]), 359.1003930340942), (tensor([ 69.4068, 561.9005]), 386.4555479237259), (tensor([510.7785, 338.3152]), 167.4361045786386), (tensor([211.7621, 350.8850]), 277.9869083233698), (tensor([1275.7411,  494.5291]), 825.838443238543

**Problema**: Cuando no detecta la muñeca en unos frames y luego la distancia es 1.8 veces mayor que el máximo valor de distancia obtenido anteriormente, se pierde el rastro de la muñeca.

**Posible solución**: hacer el umbral más permisivo cuando no se detecte en varios frames (se supone que la persona se ha movido más durante ese tiempo sin detección).

In [14]:
import torch
import cv2
import numpy as np
import math
from ultralytics import YOLO

# Model
model = YOLO('yolo11x-pose.pt')
#model=YOLO('yolov8n-pose.pt') más rápido
# Images
video = cv2.VideoCapture("videos/people2.mp4")

success, frame = video.read()
size=(frame.shape[1], frame.shape[0])
fourcc = cv2.VideoWriter_fourcc(*'DIVX')
video_out = cv2.VideoWriter('output/yolo_pose.mp4', fourcc, 30.0, size)

punto_base = None
history_dist = []
umbral = 1.8

# Bucle a través de los fotogramas del video
while (success):
    puntos =[]
    dist_ptos = []
   
    # Ejecutar seguimiento YOLOv8 en el fotograma, persistiendo los rastreos entre fotogramas
    results = model.track(frame)[0]

    for r in results:
        kpts = r.keypoints
        nk = kpts.shape[1]
        for i in range(nk):
            if i==9: # muñeca izquierda
                keypoint=kpts.xy[0,i]    
                x, y = (int(keypoint[0])),(int(keypoint[1]))
                if punto_base is None:
                    punto_base = keypoint
                    cv2.circle(frame, (x,y),5,(0,0,255),-1)
                dist_temp = math.dist(punto_base, keypoint)
                if punto_base is not None:
                    dist_ptos.append((keypoint, dist_temp))
    print(dist_ptos)
    mejor = sorted(dist_ptos, key= lambda x: x[1])[0]
    print(history_dist)
    if len(history_dist) != 0 and sum(history_dist) != 0:
        print(f"Mejor: {mejor[1]}")
        print(f"max: {max(history_dist)}")
        if mejor[1] < max(history_dist) * umbral:
            punto_base = mejor[0]
            history_dist.append(mejor[1])
            print(mejor)
            cv2.circle(frame, ((int(mejor[0][0])),(int(mejor[0][1]))),5,(0,0,255),-1)
        else:
            umbral += 0.5
    else:
        punto_base = mejor[0]
        history_dist.append(mejor[1])
        print(mejor)
        cv2.circle(frame, ((int(mejor[0][0])),(int(mejor[0][1]))),5,(0,0,255),-1)
                
    # Mostrar el fotograma anotado
    cv2.imshow('salida',frame)
    video_out.write(frame)
    success, frame = video.read()
    # Romper el bucle si se presiona 'q'
    if cv2.waitKey(1) & 0xFF == ord("q"):
         success = False
         break
    

# Liberar el objeto de captura de video y cerrar la ventana de visualización
video.release()
video_out.release()
cv2.destroyAllWindows()


0: 384x640 7 persons, 307.6ms
Speed: 0.8ms preprocess, 307.6ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)
[(tensor([449.9027, 494.2927]), 0.0), (tensor([973.1995, 505.4446]), 523.4155972519832), (tensor([ 71.5825, 562.6755]), 384.4507043742105), (tensor([775.7858, 665.6659]), 368.1964004587411), (tensor([508.8169, 342.8883]), 162.4628659589321), (tensor([212.4267, 346.5956]), 279.6592592500096), (tensor([1278.4751,  499.3896]), 828.5880944314175)]
[]
(tensor([449.9027, 494.2927]), 0.0)

0: 384x640 7 persons, 344.1ms
Speed: 1.1ms preprocess, 344.1ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)
[(tensor([967.6624, 502.2855]), 517.8214249294863), (tensor([452.8474, 488.6693]), 6.3477874052729835), (tensor([771.2878, 654.4947]), 359.1003930340942), (tensor([ 69.4068, 561.9005]), 386.4555479237259), (tensor([510.7785, 338.3152]), 167.4361045786386), (tensor([211.7621, 350.8850]), 277.9869083233698), (tensor([1275.7411,  494.5291]), 825.838443238543

Al hacer el umbral más permisivo, se obtienen errores. La muñeca se detecta en otra persona que se cruza con la objetivo.